## **PARTE 2: Impacto Financiero y JOINS (6 Ejercicios de Negocio)**

11. El Costo del Marketing Inútil (Agregación con Filtro): Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
%sql
SELECT
  COUNT(*) * 1.5 AS pago_agencia_reservas_canceladas
FROM cursodataacademy.hotel_db.tb_reservas_prueba
WHERE distribution_channel = 'TA/TO'
  AND is_canceled = 1;

12. El Costo de Cancelar Tarde (DATE_DIFF): (Considera que tienes las columnas arrival_date y reservation_status_date ambas en formato DATE). Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.


In [0]:
%sql
WITH base AS (
  SELECT *,
    TO_DATE(
      CONCAT(arrival_date_year, '-', arrival_date_month, '-', arrival_date_day_of_month),
      'yyyy-MMMM-d'
    ) AS arrival_date,
    TO_DATE(reservation_status_date, 'd/MM/yy') AS status_date
  FROM cursodataacademy.hotel_db.tb_reservas_prueba
)
SELECT
  COUNT(*) AS reservas_canceladas_tarde,
  COUNT(*) * 120 AS perdida_total_usd
FROM base
WHERE is_canceled = 1
  AND DATEDIFF(arrival_date, status_date) < 3;

13. (JOIN 1 - LEFT JOIN - Entendiendo el Modelo): Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais (ej. PRT, ESP) y nombre_pais. Une tu tabla tb_reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

In [0]:
%sql
-- CREACIÓN DE LA TABLA DE PAÍSES (vacía)
CREATE OR REPLACE TABLE cursodataacademy.hotel_db.tb_paises (
  cod_pais STRING,
  nombre_pais STRING
);

-- LEFT JOIN ENTRE RESERVAS Y PAÍSES
SELECT
  r.is_canceled,
  p.nombre_pais
FROM cursodataacademy.hotel_db.tb_reservas_prueba r
LEFT JOIN cursodataacademy.hotel_db.tb_paises p
  ON r.country = p.cod_pais;

14. (JOIN 2 - INNER JOIN con Agrupación): Usando el mismo JOIN del ejercicio anterior, averigua: ¿Cuáles son los 5 países (muestra el nombre_pais) que tienen la mayor cantidad de reservas canceladas?

In [0]:
%sql
SELECT
  p.nombre_pais,
  COUNT(*) AS reservas_canceladas
FROM cursodataacademy.hotel_db.tb_reservas_prueba r
INNER JOIN cursodataacademy.hotel_db.tb_paises p
  ON r.country = p.cod_pais
WHERE r.is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY reservas_canceladas DESC
LIMIT 5;

15. (JOIN 3 - Simulación FULL OUTER JOIN): Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor para identificar si hay reservas sin vendedor o vendedores sin reservas asignadas.

In [0]:
%sql
-- CREACIÓN DE LA TABLA DE VENDEDORES (vacía)
CREATE TABLE IF NOT EXISTS cursodataacademy.hotel_db.tb_vendedores (
  id_vendedor INT,
  nombre_vendedor STRING
);

-- AGREGAR LA COLUMNA id_vendedor A LA TABLA DE PRUEBA
ALTER TABLE cursodataacademy.hotel_db.tb_reservas_prueba
ADD COLUMNS (id_vendedor INT);

-- FULL OUTER JOIN ENTRE RESERVAS Y VENDEDORES
SELECT
  r.distribution_channel,
  v.nombre_vendedor
FROM cursodataacademy.hotel_db.tb_reservas_prueba r
FULL OUTER JOIN cursodataacademy.hotel_db.tb_vendedores v
  ON r.id_vendedor = v.id_vendedor;

16. (JOIN 4 - RIGHT JOIN): El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel (hotel).

In [0]:
%sql
-- CREACIÓN DE LA TABLA DE FACTURAS (vacía)
CREATE TABLE IF NOT EXISTS cursodataacademy.hotel_db.tb_facturas (
  id_reserva INT,
  monto_facturado DOUBLE
);

-- AGREGAR LA COLUMNA id_reserva A LA TABLA DE PRUEBA
ALTER TABLE cursodataacademy.hotel_db.tb_reservas_prueba
ADD COLUMNS (id_reserva INT);

-- RIGHT JOIN ENTRE RESERVAS Y FACTURAS
SELECT
  f.id_reserva,
  f.monto_facturado,
  r.hotel
FROM cursodataacademy.hotel_db.tb_reservas_prueba r
RIGHT JOIN cursodataacademy.hotel_db.tb_facturas f
  ON r.id_reserva = f.id_reserva;